# U-JEPA Phase 1: N-LoRA continual learning on Qwen3-14B-Instruct

Gate: average forgetting < 5 percent after sequential FOMC -> ScienceQA-text training. Requires GPU T4 x2 and Internet On. Output: `/kaggle/working/results/phase1_continual.json`.

**Note on disk**: HF model cache goes to `/tmp/hf_cache` (50+ GB ephemeral) because `/kaggle/working` is capped at 20 GB and Qwen3-14B is ~28 GB.

In [ ]:
import shutil, os
stale = '/kaggle/working/hf_cache'
if os.path.isdir(stale):
    print(f'removing stale cache at {stale}')
    shutil.rmtree(stale)
os.makedirs('/tmp/hf_cache', exist_ok=True)

In [ ]:
import subprocess, os, sys
if not os.path.exists('/kaggle/working/U-JEPA'):
    subprocess.run(['git', 'clone', 'https://github.com/kartikshirode/U-JEPA.git',
                    '/kaggle/working/U-JEPA'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/U-JEPA', 'pull'], check=True)
os.chdir('/kaggle/working/U-JEPA')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print(f'Warning: no HF_TOKEN secret found ({e})')
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['HF_HUB_CACHE'] = '/tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'

In [ ]:
import torch
print(f'torch {torch.__version__}, cuda {torch.version.cuda}, GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory // (1024**2)} MiB')

In [ ]:
import subprocess, sys, os
env = os.environ.copy()
env['HF_HOME'] = '/tmp/hf_cache'
env['HF_HUB_CACHE'] = '/tmp/hf_cache'
env['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'
subprocess.run([sys.executable, 'scripts/02_train_continual_phase1.py'], check=True, env=env)

In [ ]:
import json
from pathlib import Path
p = Path('/kaggle/working/results/phase1_continual.json')
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=2))
else:
    print('No results file yet')